Setup and Import

In [39]:
import numpy as np
import tensorflow as tf
import os
import json
import time
import pandas as pd
import sys
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import nltk
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

# Download dependensi METEOR
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

sys.path.append(os.path.abspath("../../")) 
from rnn_lstm.decoder_rnn import DecoderRNN
from rnn_lstm.decoder_lstm import DecoderLSTM

# Load metadata
with open('../../../dataset/vocab.json', 'r') as f:
    vocab_data = json.load(f)

VOCAB_SIZE      = vocab_data['vocab_size']
BASE_MAX_LENGTH = vocab_data['max_length']
WORD_TO_INDEX   = vocab_data['word_to_index']
INDEX_TO_WORD   = {int(k): v for k, v in vocab_data['index_to_word'].items()}
FEATURE_DIR     = "../../../dataset/features"
IMAGE_DIR       = "../../../dataset/image"

Helper: Universal Keras Builder and Extractor

In [40]:
def build_keras_model(num_layers, hidden_size, cell_type="rnn", embed_dim=256):
    image_input = tf.keras.Input(shape=(2048,), name="image_input")
    image_emb   = tf.keras.layers.Dense(embed_dim, activation='relu', name="proj_dense")(image_input)
    image_emb   = tf.keras.layers.Reshape((1, embed_dim))(image_emb)

    # Supaaya fleksibel -> max_length = None
    caption_input = tf.keras.Input(shape=(None,), name="caption_input")
    caption_emb   = tf.keras.layers.Embedding(VOCAB_SIZE, embed_dim, mask_zero=True, name="embedding")(caption_input)

    merged = tf.keras.layers.Concatenate(axis=1)([image_emb, caption_emb])

    x = merged
    for i in range(num_layers):
        if cell_type == "rnn":
            x = tf.keras.layers.SimpleRNN(hidden_size, return_sequences=True, name=f"rnn_{i}")(x)
        else:
            x = tf.keras.layers.LSTM(hidden_size, return_sequences=True, name=f"lstm_{i}")(x)

    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    
    output = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax'),
        name="time_dist_out"
    )(x)

    return tf.keras.Model(inputs=[image_input, caption_input], outputs=output)

def extract_weights_to_dict(keras_model, num_layers, cell_type="rnn"):
    params = {}
    params['proj_w'], params['proj_b'] = keras_model.get_layer("proj_dense").get_weights()
    params['embedding'] = keras_model.get_layer("embedding").get_weights()
    
    for i in range(num_layers):
        layer_name = f"{cell_type}_{i}"
        params[layer_name] = keras_model.get_layer(layer_name).get_weights() # [kernel, recurrent_kernel, bias]
        
    time_dist_layer = keras_model.get_layer("time_dist_out")
    params['out_w'], params['out_b'] = time_dist_layer.get_weights()
    return params

Inference and Evaluation Function

In [41]:
def generate_caption(model_scratch, image_feature, max_length, word_to_index, index_to_word):
    in_text = '<start>'
    for i in range(max_length):
        sequence = [word_to_index.get(w, 0) for w in in_text.split()]
        sequence = tf.keras.preprocessing.sequence.pad_sequences([sequence], maxlen=max_length, padding='post')
        
        img_input = np.expand_dims(image_feature, axis=0)
        
        probs = model_scratch.forward(img_input, sequence)
        
        current_len = len(in_text.split())
        yhat_idx = np.argmax(probs[0, current_len - 1, :])
        word = index_to_word.get(yhat_idx, '')
        
        if word == '<end>' or not word:
            break
        in_text += ' ' + word
        
    return in_text.replace('<start> ', '').strip()

def evaluate_max_length(model_scratch, test_df, feature_dir, max_len_variation):
    actual, predicted = [], []
    test_grouped = test_df.groupby('image')['caption'].apply(list).to_dict()
    
    for img_name, captions in list(test_grouped.items())[:100]: # mohon maap tapi kalau 1000 kelamaan 
    # for img_name, captions in list(test_grouped.items()): 
        feat_path = os.path.join(feature_dir, img_name + ".npy")
        feature = np.load(feat_path)
        
        yhat = generate_caption(model_scratch, feature, max_len_variation, WORD_TO_INDEX, INDEX_TO_WORD)
        references = [c.replace('<start> ', '').replace(' <end>', '').split() for c in captions]
        
        actual.append(references)
        predicted.append(yhat.split())
    
    smoothie = SmoothingFunction().method4
    bleu_4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    return bleu_4

Data Preparation and Splitting

In [42]:
# Data Preparation
df_captions = pd.read_csv("../../../dataset/captions.txt")
df_captions = df_captions.dropna()
df_captions['caption'] = "<start> " + df_captions['caption'].str.lower() + " <end>"

# Splitting
unique_images = df_captions['image'].unique()
train_val_imgs, test_imgs = train_test_split(unique_images, test_size=1000, random_state=42)
test_df = df_captions[df_captions['image'].isin(test_imgs)]

print(f"Test Set disiapkan: {len(test_imgs)} gambar unik.")

Test Set disiapkan: 1000 gambar unik.


Best Model

In [43]:
weight_dir = "../weights/"
best_models = {"rnn": {"score": -1, "file": ""}, "lstm": {"score": -1, "file": ""}}

for file in os.listdir(weight_dir):
    if file.endswith("_metadata.json"):
        filepath = os.path.join(weight_dir, file)
        with open(filepath, 'r') as f:
            meta = json.load(f)
            
        cell_type = "rnn" if "rnn" in file else "lstm"
        score = meta['bleu_4'] 
        
        if score > best_models[cell_type]["score"]:
            best_models[cell_type]["score"] = score
            best_models[cell_type]["file"] = file.replace("_metadata.json", "")
            best_models[cell_type]["num_layers"] = meta["num_layers"]
            best_models[cell_type]["hidden_size"] = meta["hidden_size"]

print("\n--- Model Terbaik Ditemukan ---")
print(f"RNN Terbaik : {best_models['rnn']['file']}  (BLEU-4: {best_models['rnn']['score']:.4f})")
print(f"LSTM Terbaik: {best_models['lstm']['file']} (BLEU-4: {best_models['lstm']['score']:.4f})")


--- Model Terbaik Ditemukan ---
RNN Terbaik : rnn_l1_h512  (BLEU-4: 0.1075)
LSTM Terbaik: lstm_l1_h512 (BLEU-4: 0.0817)


Comparison 1: Pengaruh MAX_LENGTH

In [44]:
length_variations = [3, 5, 10]
length_results = []

for cell_type in ["rnn", "lstm"]:
    best_config = best_models[cell_type]
    
    # Scratch Model Terbaik
    tf.keras.backend.clear_session()
    k_model = build_keras_model(best_config['num_layers'], best_config['hidden_size'], cell_type)
    k_model.load_weights(os.path.join(weight_dir, f"{best_config['file']}.weights.h5"))
    weights_dict = extract_weights_to_dict(k_model, best_config['num_layers'], cell_type)
    
    if cell_type == "rnn":
        scratch_model = DecoderRNN(VOCAB_SIZE, 256, best_config['hidden_size'], best_config['num_layers'])
    else:
        scratch_model = DecoderLSTM(VOCAB_SIZE, 256, best_config['hidden_size'], best_config['num_layers'])
        
    scratch_model.set_model_params(weights_dict)
    
    # Evaluasi variasi max_length
    for m_len in length_variations:
        start_t = time.time()
        b4_score = evaluate_max_length(scratch_model, test_df, FEATURE_DIR, m_len)
        exec_t = time.time() - start_t
        
        print(f"Cell Type: {cell_type.upper()} | Max Length: {m_len:2d} | BLEU-4: {b4_score:.4f} | Waktu: {exec_t:.2f}s")
        length_results.append({
            "Arsitektur": cell_type.upper(),
            "Max Length": m_len,
            "BLEU-4": round(b4_score, 4)
        })

Cell Type: RNN | Max Length:  3 | BLEU-4: 0.0234 | Waktu: 104.07s
Cell Type: RNN | Max Length:  5 | BLEU-4: 0.0974 | Waktu: 16.93s
Cell Type: RNN | Max Length: 10 | BLEU-4: 0.0973 | Waktu: 31.65s
Cell Type: LSTM | Max Length:  3 | BLEU-4: 0.0172 | Waktu: 18.81s
Cell Type: LSTM | Max Length:  5 | BLEU-4: 0.0769 | Waktu: 37.71s
Cell Type: LSTM | Max Length: 10 | BLEU-4: 0.0763 | Waktu: 350.49s


In [45]:
print("\nREKAP PENGARUH MAX_LENGTH (100 data test):")
print(pd.DataFrame(length_results).to_string(index=False))


REKAP PENGARUH MAX_LENGTH (100 data test):
Arsitektur  Max Length  BLEU-4
       RNN           3  0.0234
       RNN           5  0.0974
       RNN          10  0.0973
      LSTM           3  0.0172
      LSTM           5  0.0769
      LSTM          10  0.0763


Comparison 2: RNN vs LSTM (qualitative)

In [46]:
print("\n" + "="*50)
print("INFERENSI KUALITATIF (10 GAMBAR ACAK) - l1_h256")
print("="*50)

# Prepare model scratch
models_kualitatif = {}
for c_type in ["rnn", "lstm"]:
    tf.keras.backend.clear_session()
    k_model = build_keras_model(num_layers=1, hidden_size=256, cell_type=c_type)
    k_model.load_weights(os.path.join(weight_dir, f"{c_type}_l1_h256.weights.h5"))
    w_dict = extract_weights_to_dict(k_model, 1, c_type)
    
    if c_type == "rnn":
        models_kualitatif['rnn'] = DecoderRNN(VOCAB_SIZE, 256, 256, 1)
    else:
        models_kualitatif['lstm'] = DecoderLSTM(VOCAB_SIZE, 256, 256, 1)
        
    models_kualitatif[c_type].set_model_params(w_dict)

# 10 gambar unik acak
np.random.seed(42)
test_grouped = test_df.groupby('image')['caption'].apply(list).to_dict()
all_test_images = list(test_grouped.keys())
random_10_images = np.random.choice(all_test_images, 10, replace=False)

for idx, img_name in enumerate(random_10_images):
    print(f"\n[{idx+1}/10] File Gambar: {img_name}")
    
    # Load feature
    feat_path = os.path.join(FEATURE_DIR, img_name + ".npy")
    feature = np.load(feat_path)
    
    # Generate Caps
    rnn_cap = generate_caption(models_kualitatif['rnn'], feature, BASE_MAX_LENGTH, WORD_TO_INDEX, INDEX_TO_WORD)
    lstm_cap = generate_caption(models_kualitatif['lstm'], feature, BASE_MAX_LENGTH, WORD_TO_INDEX, INDEX_TO_WORD)
    
    # Ground Truths
    gts = [c.replace('<start> ', '').replace(' <end>', '') for c in test_grouped[img_name]]
    
    img_path = os.path.join(IMAGE_DIR, img_name)
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
    
    print(f"Caption RNN  : {rnn_cap}")
    print(f"Caption LSTM : {lstm_cap}")
    print(f"Caption Asli :")
    for i, gt in enumerate(gts):
        print(f"   {i+1}. {gt}")


INFERENSI KUALITATIF (10 GAMBAR ACAK) - l1_h256


d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(



[1/10] File Gambar: 3139238055_2817a0c7d8.jpg
Caption RNN  : a man in a blue shirt is jumping
Caption LSTM : a man is in a red shirt
Caption Asli :
   1. four people make a silly pose at the dinner table .
   2. four people smile and hold golden plates behind their heads .
   3. friends posing for a picture while holding plates
   4. two men and two women hold gold plates behind their heads .
   5. two men and women pose for a camera with large gold plates held behind their heads .

[2/10] File Gambar: 3485486737_953f9d3be2.jpg
Caption RNN  : a man in a blue shirt is jumping
Caption LSTM : a man is in a red shirt
Caption Asli :
   1. a man in the back of a van takes a picture of a yellow truck .
   2. a man looking out the back of his car looking at a yellow van behind
   3. a person riding in the back of one vehicle is looking down the road towards a yellow truck .
   4. photographer shooting picture of yellow bus .
   5. two people sit in shadow in the open back of a vehicle followe